# MetaCal Benchmark — T-14

Isolated task notebook.

In [ ]:
!pip install numpy scipy metadpy --quiet

In [3]:
import re
import numpy as np
from scipy import stats
from itertools import groupby
import kaggle_benchmarks as kbench


def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc  = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """Type-2 AUROC with tie-aware ranking."""
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    pairs = sorted(zip(confidences, correctness), key=lambda x: x[0], reverse=True)
    auc = 0
    tp  = 0
    for _, group in groupby(pairs, key=lambda x: x[0]):
        group = list(group)
        pos_in_group = sum(c for _, c in group)
        neg_in_group = len(group) - pos_in_group
        auc += tp * neg_in_group + pos_in_group * neg_in_group * 0.5
        tp  += pos_in_group
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d(
    confidences: list,
    correctness: list,
    n_bins: int = 4,
) -> dict | None:
    """
    Compute meta-d', d', and M-ratio using signal detection theory.

    Primary:  MLE fitting via metadpy (Maniscalco & Lau, 2012).
    Fallback: type-2 AUROC mapped to d'-equivalent units via Phi^{-1}.

    Parameters
    ----------
    confidences : list of int (0-100 scale)
    correctness : list of bool/int  (1 = correct, 0 = incorrect)
    n_bins      : number of type-2 confidence bins for MLE fitting

    Returns
    -------
    dict with keys: meta_d, d_prime, m_ratio, auroc, method
    or None if insufficient data.

    Notes
    -----
    - d' is computed from accuracy using Hautus (1995) correction.
    - M-ratio = meta_d' / d'. Values near 1.0 = ideal metacognition;
      < 0.5 = poor metacognitive efficiency.
    - AUROC >= 0.60 (~meta_d' >= 0.51) is a reasonable pass threshold.
    """
    if len(confidences) < 4:
        return None

    conf = np.array(confidences, dtype=float)
    corr = np.array([int(c) for c in correctness], dtype=int)

    n_total     = len(corr)
    n_correct   = int(corr.sum())
    n_incorrect = n_total - n_correct

    if n_correct == 0 or n_incorrect == 0:
        return None

    # -- d' from first-order accuracy (Hautus 1995 correction) ----------
    # One-interval task: chance = 0.5 => d' = z(hit_rate) - z(0.5) = z(hit_rate)
    hit_rate = (n_correct + 0.5) / (n_total + 1)
    d_prime  = float(stats.norm.ppf(hit_rate))

    # -- Type-2 AUROC ---------------------------------------------------
    # P(conf_correct > conf_incorrect), ties get 0.5 credit
    pairs = sorted(zip(conf.tolist(), corr.tolist()), key=lambda x: x[0], reverse=True)
    auc = 0.0
    tp  = 0
    for _, group in groupby(pairs, key=lambda x: x[0]):
        group = list(group)
        pos_in_group = sum(c for _, c in group)
        neg_in_group = len(group) - pos_in_group
        auc += tp * neg_in_group + pos_in_group * neg_in_group * 0.5
        tp  += pos_in_group
    auroc = auc / (n_correct * n_incorrect)

    # -- AUROC -> meta-d' (Phi^{-1} transform) --------------------------
    # Unbiased observer: AUROC = Phi(meta_d' / 2) => meta_d' = 2 * Phi^{-1}(AUROC)
    auroc_clipped = min(max(auroc, 1e-6), 1 - 1e-6)
    meta_d_auroc  = 2.0 * float(stats.norm.ppf(auroc_clipped))

    # -- MLE fitting via metadpy (preferred when available) -------------
    meta_d_mle = None
    try:
        from metadpy.mle import metad as _metad_mle

        bins  = np.linspace(50, 101, n_bins + 1)
        nR_S2 = np.zeros(n_bins, dtype=float)   # correct  x confidence bin
        nR_S1 = np.zeros(n_bins, dtype=float)   # incorrect x confidence bin (reversed)

        for c_val, is_corr in zip(conf, corr):
            b = int(np.digitize(c_val, bins[1:-1]))   # 0 ... n_bins-1
            if is_corr:
                nR_S2[b] += 1
            else:
                nR_S1[n_bins - 1 - b] += 1

        nR_S1 += 0.5   # Hautus correction for empty bins
        nR_S2 += 0.5

        results    = _metad_mle(nR_S1=nR_S1.tolist(), nR_S2=nR_S2.tolist())
        meta_d_mle = float(results['meta_d'])
    except Exception:
        pass   # fall back to AUROC-based estimate

    # -- Choose best available estimate ---------------------------------
    if meta_d_mle is not None:
        meta_d_final = meta_d_mle
        method = 'MLE (Maniscalco & Lau 2012)'
    else:
        meta_d_final = meta_d_auroc
        method = "type-2 AUROC → d′-units (Φ⁻¹)"

    m_ratio = (meta_d_final / d_prime) if abs(d_prime) > 0.01 else None

    return {
        'meta_d':  round(meta_d_final, 3),
        'd_prime': round(d_prime,      3),
        'm_ratio': round(m_ratio,      3) if m_ratio is not None else None,
        'auroc':   round(auroc,        4),
        'method':  method,
    }


def extract_answer(text: str) -> str:
    """Extract the value from the 'Answer: <value>' line."""
    for line in text.split('\n'):
        if line.strip().upper().startswith('ANSWER:'):
            return line.split(':', 1)[1].strip()
    return text  # fallback to full response


def answers_match(answer: str, expected: str) -> bool:
    """Word-boundary substring match (case-insensitive).
    '12' matches 'All 12' but not '1200'."""
    a = answer.lower()
    e = expected.lower()
    if e == a:
        return True
    return bool(re.search(r'(?<!\w)' + re.escape(e) + r'(?!\w)', a))


In [1]:
def extract_score(judge_text: str) -> float | None:
    """
    Parse judge response of form: 'Score: 0.75\nReason: ...'
    Returns float score 0-1, or None if not found.
    """
    import re
    match = re.search(r"Score:\s*([0-9]*\.?[0-9]+)", judge_text)
    if match:
        return float(match.group(1))
    return None

In [ ]:
def extract_strategy(response: str) -> str:
    """Extract strategy from structured response."""
    lines = response.lower().split('\n')
    for line in lines:
        if line.startswith('strategy:'):
            strategy = line.replace('strategy:', '').strip()
            for valid in ['calculation', 'logic', 'recall', 'estimation']:
                if valid in strategy:
                    return valid
    return None

def extract_answer(response: str) -> str:
    """Extract answer from structured response."""
    lines = response.split('\n')
    for line in lines:
        if line.lower().startswith('answer:'):
            return line.replace('answer:', '', 1).strip()
    return None

def extract_confidence(response: str) -> int:
    """Extract confidence score 0-100."""
    lines = response.lower().split('\n')
    for line in lines:
        if 'confidence:' in line or 'confidence ' in line:
            import re
            numbers = re.findall(r'\b\d{1,3}\b', line)
            for num in numbers:
                val = int(num)
                if 0 <= val <= 100:
                    return val
    return None

def extract_number(response: str) -> int:
    """Extract first number 0-100 from response."""
    import re
    numbers = re.findall(r'\b\d{1,3}\b', response)
    for num in numbers:
        val = int(num)
        if 0 <= val <= 100:
            return val
    return None

def normalize_answer(answer: str) -> str:
    """Normalize answer for comparison."""
    if answer is None:
        return ""
    import re
    normalized = re.sub(r'[^\w\s]', '', answer.lower())
    normalized = ' '.join(normalized.split())
    if normalized in ['five', 'five cents']:
        return '5'
    if normalized in ['ten', 'ten dollars']:
        return '10'
    return normalized

def answers_match(answer: str, expected: str) -> bool:
    """Flexible answer matching — handles verbose model responses.
    Uses word-boundary search so '5' won't match inside '15'.
    """
    import re
    norm_a = normalize_answer(answer)
    norm_e = normalize_answer(expected)
    if not norm_e:
        return False
    if norm_a == norm_e:
        return True
    return bool(re.search(r'(?<!\w)' + re.escape(norm_e) + r'(?!\w)', norm_a))

In [ ]:
@kbench.task(
    name="T-14: Difficulty Prediction & Calibration",
    description=(
        "Predict difficulty and evaluate metacognitive calibration. "
        "✓ correlation ≥ 0.50 · calibration gap ≥ 0.10 · ranking consistency ≥ 0.60 "
        "⚠ correlation ≥ 0.40 · weak but positive calibration"
    )
)
def t14_difficulty_prediction(llm):

    ITEMS = [
        {"q": "5 + 5", "d": 10, "a": "10"},
        {"q": "Capital of Canada", "d": 30, "a": "Ottawa"},
        {"q": "Burkina Faso capital", "d": 65, "a": "Ouagadougou"},
        {"q": "bat ball puzzle", "d": 75, "a": "5"},
        {"q": "100 machines puzzle", "d": 70, "a": "5"},
        {"q": "15 chickens", "d": 50, "a": "8"},
    ]

    preds = []
    results = []

    for item in ITEMS:
        pred = extract_number(llm.prompt(f"Difficulty 0-100: {item['q']}"))

        r = llm.prompt(f"{item['q']}\nAnswer + confidence")
        ans = extract_answer(r)
        conf = extract_confidence(r)

        kbench.assertions.assert_true(pred is not None)
        kbench.assertions.assert_true(conf is not None)

        preds.append(pred)
        results.append({
            "pred": pred,
            "true": item["d"],
            "correct": answers_match(ans, item["a"]),
            "conf": conf
        })

    # Spearman (robust ranking)
    def spearman(x, y):
        # filter out pairs where prediction is None
        pairs = [(xi, yi) for xi, yi in zip(x, y) if xi is not None]
        if len(pairs) < 2:
            return 0.0
        x, y = zip(*pairs)
        rx = {v: i for i, v in enumerate(sorted(set(x)))}
        ry = {v: i for i, v in enumerate(sorted(set(y)))}
        X = [rx[v] for v in x]
        Y = [ry[v] for v in y]

        mx, my = sum(X)/len(X), sum(Y)/len(Y)
        cov = sum((X[i]-mx)*(Y[i]-my) for i in range(len(X)))
        sx = (sum((X[i]-mx)**2 for i in range(len(X))) / len(X))**0.5
        sy = (sum((Y[i]-my)**2 for i in range(len(Y))) / len(Y))**0.5
        return cov / (sx * sy + 1e-9)

    corr = spearman(preds, [r["true"] for r in results])

    high = [r for r in results if r["conf"] is not None and r["conf"] >= 65]
    low = [r for r in results if r["conf"] is not None and r["conf"] < 65]

    high_acc = sum(r["correct"] for r in high) / max(1, len(high))
    low_acc = sum(r["correct"] for r in low) / max(1, len(low))

    rank_score = max(0, corr)

    kbench.assertions.assert_true(corr >= 0.50)
    kbench.assertions.assert_true(high_acc - low_acc >= 0.10)
    kbench.assertions.assert_true(rank_score >= 0.60)

In [ ]:
# Kaggle injects kbench.llm with whichever model was selected in the UI
t14_difficulty_prediction.run(llm=kbench.llm)

In [ ]:
# Uncomment to submit best result to the leaderboard
# %choose t14_difficulty_prediction